# Parameterising a pyISSM model — a step-by-step sorkflow

This notebook walks you through the parameterisation step for a Pine Island Glacier (PIG) setup using **pyISSM**, the Python interface for the Ice Sheet System Model (ISSM).

**No prior experience with ISSM is assumed.** Each concept is explained in plain English before you run any code.

---

## What you will do in this notebook

1. Set up your working environment and directories
2. Build a velocity-adapted mesh of the Pine Island Glacier domain
3. Parameterise the model (set up some of the physical fields ISSM needs, as an example)
---

> **How to use this notebook:**
> - **Cells like this one** (formatted text) are explanations. Read them before moving on.
> - **Code cells** (grey boxes with `[ ]:` on the left) are commands you run. Click on them and press **Shift + Enter**, or click the **Run** button in the toolbar.
> - Work through the steps **in order**. Later cells depend on earlier ones.

> **Before starting:** Make sure you have completed the pyISSM installation and meshing tutorial notebooks, and that `import pyissm` runs without errors.

---

# PART 1 — Setup

---

## Step 1: Import packages and set up directories

We begin by importing the Python packages we need and defining the paths to our working directories.

**Update the paths below to point to your own project folder on Gadi.**

In [ ]:
import sys
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio

# Point Python to the pyISSM source if it is not installed system-wide
sys.path.insert(0, '/g/data/<project>/<userid>/pyISSM/src')  # <- update if needed

import pyissm
import ccdtools as dp

print('pyISSM imported successfully!')
print('pyISSM version:', pyissm.__version__)
print('Loading from:', pyissm.__file__)

In [ ]:
# ── Update these paths to your own directories on Gadi ───────────────────────
tutorial_dir  = '/g/data/<project>/<userid>/pyISSM/tutorials'   # <- change this
execution_dir = '/g/data/<project>/<userid>/PIG'   # <- change this
# ─────────────────────────────────────────────────────────────────────────────

asset_dir  = tutorial_dir  + '/assets'
model_dir  = execution_dir + '/Models'
plots_dir  = execution_dir + '/Plots'
exp_dir    = execution_dir + '/Exp'
results_dir = execution_dir + '/Results/InitialN/Restart'

# Create output directories if they don't already exist
for d in [model_dir, plots_dir, results_dir]:
    os.makedirs(d, exist_ok=True)

# Set paths to key input files
domain_file   = exp_dir + '/PineIslandDomain.exp'
param_file    = execution_dir + '/Par/PineIslandParam.py'

print(f'execution_dir : {execution_dir}')
print(f'model_dir     : {model_dir}')
print(f'Domain file   : {domain_file}')

> **What are these directories for?**
> - `execution_dir` is the root of your project — all inputs and outputs live under here
> - `model_dir` is where model `.nc` files are saved and loaded between steps
> - `exp_dir` contains `.exp` domain outline files (explained in the meshing step)
> - `results_dir` is where intermediate restart fields from the GlaDS spin-up are saved

In [ ]:
# Load the ACCESS Cryosphere Data Catalogue
# This gives us access to observational datasets (BedMachine, MEaSUREs velocity, RACMO, etc.)
catalog = dp.catalog.DataCatalog()
print('Data catalogue loaded.')

## Step 2: Choose which steps to run

This notebook is divided into discrete steps. Rather than running every step every time you open the notebook, you control which steps are active by adding their names to the `steps` list below.

This mirrors how large ISSM workflows are typically structured — you build up the model progressively, saving after each step, so you can reload and continue without repeating expensive computations.

Available steps:

| Step name | What it does |
|---|---|
| `mesh` | Generate the velocity-adapted mesh |
| `param` | Parameterise the model (load all physical fields) |

In [ ]:
# ── Edit this list to control which steps are active ─────────────────────────
steps = ['mesh', 'param']
# ─────────────────────────────────────────────────────────────────────────────

print('Active steps:', steps)

---

# PART 2 — Meshing

---

## Step 3: Generate the Pine Island Glacier mesh

Before ISSM can solve any equations, it needs a mesh: a triangulated representation of the ice sheet domain. If you haven't already worked through the meshing tutorial, we recommend doing so before continuing here. This step assumes familiarity with `triangle` and `bamg`.

For Pine Island Glacier, we use a **velocity-adapted mesh** generated in two refinement passes:

1. A coarse initial mesh is created from the domain outline file (`PineIslandDomain.exp`) at 5 km resolution using `triangle`
2. We interpolate surface velocity and the BedMachine ice mask onto the mesh
3. We use `bamg` to refine the mesh so that elements are smaller where ice flows fast and where the geometry is complex, and coarser in the slow-moving interior
4. Steps 2–3 are repeated a second time, so the mesh adapts to velocity interpolated onto the improved triangulation

The key refinement rules are:
- Fast-flowing grounded ice (velocity > 50 m/yr): maximum element size capped at **1500 m**
- Floating ice shelves: minimum element size of **500 m**
- Global bounds: minimum 50 m, maximum 50 km

In [ ]:
if 'mesh' in steps:
    print('-------------------------------------------------------------')
    print(' STEP 1: Generating mesh')
    print('-------------------------------------------------------------')

    # Create a new, empty model object
    md = pyissm.model.Model()

    # Build the initial coarse mesh from the domain outline at 5 km resolution
    print('-- Building initial 5 km mesh from domain outline...')
    md = pyissm.model.mesh.triangle(md,
                                    domain_name=domain_file,
                                    resolution=5e3)

    # Load datasets from the ACCESS Cryosphere Data Catalogue
    print('-- Loading BedMachine Antarctica v3...')
    bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version='v3')

    print('-- Loading MEaSUREs InSAR velocity...')
    velocity_data = catalog.load_dataset(
        'measures_insar_based_antarctica_ice_velocity_map', version='v2')

    # Run interpolation
    # Interpolate the BedMachine ice mask and MEaSUREs velocity onto mesh node positions
    mask = pyissm.data.interp.xr_to_mesh(bedmachine_data, 'mask', md.mesh.x, md.mesh.y)
    vx   = pyissm.data.interp.xr_to_mesh(velocity_data,   'VX',   md.mesh.x, md.mesh.y)
    vy   = pyissm.data.interp.xr_to_mesh(velocity_data,   'VY',   md.mesh.x, md.mesh.y)
    vel  = np.sqrt(vx**2 + vy**2)

    # Set velocity to zero where data is missing or outside the ice sheet
    # BedMachine mask values: 0=ocean, 1=ice-free land, 2=grounded ice, 3=floating ice
    vel[np.isnan(vel) | (mask < 2)] = 0

    # Build per-node resolution arrays (NaN means no constraint at that node)
    hmaxv = np.full(md.mesh.numberofvertices, np.nan)
    hminv = np.full(md.mesh.numberofvertices, np.nan)

    # Fast-flowing grounded ice: cap maximum element size at 2000 m
    pos_fast  = (vel > 50) & (mask == 2)
    hmaxv[pos_fast] = 2000

    # Floating ice shelves: enforce minimum element size of 2000 m
    pos_float = (mask == 3)
    hminv[pos_float] = 2000

    print('   -- Remeshing with bamg...')
    md = pyissm.model.mesh.bamg(md,
                                field=vel,
                                err=1,
                                hmin=50,
                                hmax=50e3,
                                hmaxVertices=hmaxv,
                                hminVertices=hminv,
                                maxnbv=2e6,
                                gradation=1.2)

    print(f'   Mesh generation complete: {md.mesh.numberofvertices} nodes, '
          f'{md.mesh.numberofelements} elements')

    # Assign lat/lon coordinates and EPSG code
    [md.mesh.lat, md.mesh.long] = pyissm.tools.general.xy_to_ll(md.mesh.x, md.mesh.y, -1)
    md.mesh.epsg = 3031  # WGS 84 / Antarctic Polar Stereographic

    # Plot the mesh
    fig, ax = pyissm.plot.plot_mesh2d(md, color='steelblue', linewidth=0.3)
    ax.set_xlabel('X Coordinate (m)')
    ax.set_ylabel('Y Coordinate (m)')
    ax.set_title('Pine Island Glacier mesh — velocity-adapted, 2 refinement passes')
    plt.savefig(f'{plots_dir}/pineisland-mesh.pdf')
    plt.show()

    # Save the meshed model
    save_path = f'{model_dir}/pineisland-mesh.nc'
    pyissm.model.io.save_model(md, save_path)
    print(f'\nModel saved to: {save_path}')
    print(f'To reload: md = pyissm.model.io.load_model("{save_path}")')

---

# PART 3 — Parameterisation

---

## What is parameterisation?

Once you have a mesh, ISSM needs to know about the **physical state of the ice sheet** before it can solve any equations. This process of populating the model object with real-world data is called **parameterisation**.

Think of the mesh as the skeleton of the model. Parameterisation is the process of adding the flesh: the geometry of the ice sheet, what the ice is made of, how fast it's moving, what's happening at its base, and so on. Without this information, ISSM has no way to compute anything meaningful.

### Why do we need to parameterise?

ISSM can solve several different types of equations depending on what you want to simulate:

- **Stress balance**: solves for ice velocity given the current geometry, rheology, and basal friction
- **Mass transport**: evolves ice thickness through time given velocities and surface/basal mass balance
- **Thermal**: solves for the temperature field through the ice, which controls ice viscosity
- **Hydrology (GlaDS)**: simulates the subglacial drainage system, which controls basal friction
- **Transient**: runs any or all of the above together through time, often with grounding line evolution

Each of these solutions depends on different parts of the model object being populated. The table below lists fields we typically parameterise, why it is needed, and which solutions depend on it.

### Parameterisation fields reference

| Field | What it contains | Why we need it | Used by |
|---|---|---|---|
| `md.mask.ice_levelset` | Signed distance function: negative inside ice, positive outside, and 0 at the ice-front | Tells ISSM where ice exists; governs which elements are active | All solutions |
| `md.mask.ocean_levelset` | Signed distance function: negative where ocean is present, and 0 at the grounding line | Identifies floating ice and the grounding line | Stress balance, mass transport, transient |
| `md.geometry.surface` | Ice surface elevation (m above sea level) | Drives the gravitational driving stress | Stress balance, thermal, transient |
| `md.geometry.bed` | Bedrock elevation (m above sea level) | Sets the lower boundary; controls grounding | Stress balance, hydrology, transient |
| `md.geometry.thickness` | Ice thickness (m) | Central geometric variable in all equations | All solutions |
| `md.geometry.base` | Elevation of the base of the ice (m) | The base is not always equal to the bed (floating ice) | Stress balance, hydrology, transient |
| `md.materials.rheology_B` | Ice flow law parameter B (Pa s^1/3) | Controls how easily ice deforms; varies with temperature | Stress balance, thermal |
| `md.materials.rheology_n` | Flow law exponent (usually 3) | Sets the non-linearity of ice flow | Stress balance |
| `md.materials.rheology_law` | Name of the flow law (e.g. Cuffey) | Selects which formulation to use | Stress balance |
| `md.friction.coefficient` | Basal drag coefficient | Resists basal sliding; controls ice velocity at the bed | Stress balance, transient |
| `md.friction.p`, `md.friction.q` | Friction law exponents | Shape the velocity-dependence of basal drag | Stress balance |
| `md.friction.effective_pressure` | Effective pressure at the bed (N - P_w) | Couples the hydrology model to basal friction in the Budd law | Stress balance, hydrology |
| `md.inversion.vx_obs`, `md.inversion.vy_obs` | Observed surface velocity components (m/yr) | Used as a target in inversions; provides initial velocity state | Inversion, stress balance |
| `md.inversion.vel_obs` | Observed speed (m/yr) | Used directly in friction initialisation and inversions | Inversion, stress balance |
| `md.initialization.vx`, `md.initialization.vy` | Initial velocity field (m/yr) | Starting point for the solver | Stress balance, transient |
| `md.initialization.temperature` | Initial ice temperature field (K) | Starting point for the thermal solver | Thermal, transient |
| `md.initialization.watercolumn` | Initial subglacial water sheet thickness (m) | Starting point for the hydrology solver | Hydrology |
| `md.initialization.hydraulic_potential` | Initial hydraulic potential (Pa) | Starting point for the GlaDS solver | Hydrology |
| `md.smb.mass_balance` | Surface mass balance (m ice eq./yr) | Net accumulation/ablation at the surface | Mass transport, transient |
| `md.basalforcings.groundedice_melting_rate` | Basal melt rate under grounded ice (m/yr) | Geothermal and frictional melting at the bed | Mass transport, hydrology |
| `md.basalforcings.floatingice_melting_rate` | Basal melt rate under floating ice (m/yr) | Ocean-driven melting of ice shelves | Mass transport, transient |
| `md.basalforcings.geothermalflux` | Geothermal heat flux (W/m²) | Heats the base of the ice; influences basal melting and temperature | Thermal, hydrology |
| `md.thermal.spctemperature` | Temperature boundary condition at the surface (K) | The ice surface is assumed to be at the air temperature | Thermal |
| `md.stressbalance.spcvx`, `spcvy`, `spcvz` | Velocity boundary conditions | Constrain velocity at the ice margin | Stress balance, transient |

### How parameterisation is structured in this workflow

We split parameterisation into two separate `.py` files that are run in sequence using `pyissm.model.param.parameterize()`:

- **`PineIslandParam.py`** (`param_file`): Populates geometry, mask, observed velocity, geothermal heat flux, and basal melt. This is the foundation that GlaDS (the hydrology model) needs. There will be a lot of other fields that you will most likely eventually need to run a model, including choices made about the friction law, rheology, and climate forcing. We will deal will those in a separate tutorial.

In [ ]:
if 'param' in steps:
    print('-------------------------------------------------------------')
    print(' STEP 2: Parameterising model')
    print('-------------------------------------------------------------')

    # Load the mesh we created in Step 1
    print('-- Loading mesh...')
    md = pyissm.model.io.load_model(f'{model_dir}/pineisland-mesh.nc')

    # Set the flow equation to SSA (Shelfy-Stream Approximation)
    # SSA is appropriate for fast-flowing ice and ice shelves; it treats the ice as a thin viscous sheet
    print('-- Setting SSA flow equation...')
    md = pyissm.model.param.set_flow_equation(md, SSA='all')

    # Run the first parameterisation script
    # This populates: mask, geometry, velocity, geothermal heat flux, basal melt, friction placeholders
    print(f'-- Running first parameterisation script: {param_file}...')
    md = pyissm.model.param.parameterize(md, param_file)

    # Quick sanity checks on the parameterised fields
    print('\n-- Sanity checks:')
    print(f'   Surface elevation : min={np.nanmin(md.geometry.surface):.1f} m, max={np.nanmax(md.geometry.surface):.1f} m')
    print(f'   Bed elevation     : min={np.nanmin(md.geometry.bed):.1f} m, max={np.nanmax(md.geometry.bed):.1f} m')
    print(f'   Ice thickness     : min={np.nanmin(md.geometry.thickness):.1f} m, max={np.nanmax(md.geometry.thickness):.1f} m')
    print(f'   Observed velocity : min={np.nanmin(md.inversion.vel_obs):.1f} m/yr, max={np.nanmax(md.inversion.vel_obs):.1f} m/yr')
    print(f'   GHF               : min={np.nanmin(md.basalforcings.geothermalflux):.4f} W/m2, max={np.nanmax(md.basalforcings.geothermalflux):.4f} W/m2')
    print(f'   Basal melt        : min={np.nanmin(md.basalforcings.groundedice_melting_rate):.4f} m/yr, max={np.nanmax(md.basalforcings.groundedice_melting_rate):.4f} m/yr')

    # Plot all parameterised fields for a visual check
    fig, ax = plt.subplots(2, 4, constrained_layout=True, figsize=(16, 6))
    pyissm.plot.plot_model_field(md, md.mask.ocean_levelset,                     ax=ax[0,0], log=False, vmin=-1, vmax=1,  cmap='RdBu',    show_cbar=True, cbar_kwargs={'label': 'Ocean levelset'})
    pyissm.plot.plot_model_field(md, md.mask.ice_levelset,                       ax=ax[0,1], log=False, vmin=-1, vmax=1,  cmap='RdBu',    show_cbar=True, cbar_kwargs={'label': 'Ice levelset'})
    pyissm.plot.plot_model_field(md, md.geometry.thickness,                      ax=ax[0,2], log=False,                   cmap='viridis', show_cbar=True, cbar_kwargs={'label': 'Thickness (m)'})
    pyissm.plot.plot_model_field(md, md.geometry.bed,                            ax=ax[0,3], log=False,                   cmap='viridis', show_cbar=True, cbar_kwargs={'label': 'Bed elevation (m)'})
    pyissm.plot.plot_model_field(md, md.basalforcings.floatingice_melting_rate,  ax=ax[1,0], log=False,                   cmap='viridis', show_cbar=True, cbar_kwargs={'label': 'Shelf melt (m/yr)'})
    pyissm.plot.plot_model_field(md, md.basalforcings.groundedice_melting_rate,  ax=ax[1,1], log=False,                   cmap='viridis', show_cbar=True, cbar_kwargs={'label': 'Grounded melt (m/yr)'})
    pyissm.plot.plot_model_field(md, md.basalforcings.geothermalflux,            ax=ax[1,2], log=False,                   cmap='viridis', show_cbar=True, cbar_kwargs={'label': 'GHF (W/m2)'})
    pyissm.plot.plot_model_field(md, md.inversion.vel_obs,                       ax=ax[1,3], log=True,                    cmap='Spectral',show_cbar=True, cbar_kwargs={'label': 'Velocity (m/yr)'})
    ax[0,0].set_title('Ocean levelset');  ax[0,1].set_title('Ice levelset');
    ax[0,2].set_title('Thickness');       ax[0,3].set_title('Bed elevation')
    ax[1,0].set_title('Ice shelf melt');  ax[1,1].set_title('Grounded ice melt')
    ax[1,2].set_title('Geothermal flux'); ax[1,3].set_title('Observed velocity')
    plt.savefig(f'{plots_dir}/pineisland-param-fields.pdf')
    plt.show()

    # Save the parameterised model
    save_path = f'{model_dir}/pineisland-param.nc'
    pyissm.model.io.save_model(md, save_path)
    print(f'\nModel saved to: {save_path}')
    print(f'To reload: md = pyissm.model.io.load_model("{save_path}")')

---

# What comes next?

You have now completed the first parameterisation file. The next step in this workflow would be to initialise and then run a subglacial hydrology model, but we're not going to do that yet!

---

## Quick reference

| Task | Function |
|---|---|
| Save a model | `pyissm.model.io.save_model(md, path)` |
| Load a model | `md = pyissm.model.io.load_model(path)` |
| Run parameterisation script | `md = pyissm.model.param.parameterize(md, script_path)` |
| Set flow equation | `md = pyissm.model.param.set_flow_equation(md, SSA='all')` |

---

## Troubleshooting

| Problem | What to check |
|---|---|
| `ModuleNotFoundError: pyissm` | Check `sys.path.insert(0, ...)` points to the correct pyISSM `src` directory |
| `PermissionError` on save | Check the model directory exists and you have write access; run `os.makedirs(model_dir, exist_ok=True)` |
| `catalog.load_dataset` fails | Confirm the dataset name with `catalog.search('keyword')` and check your storage allocation includes the relevant project |

---

## Useful links

- **ISSM documentation**: https://issmteam.github.io/ISSM-Documentation/
- **pyISSM documentation**: https://pyissm.readthedocs.io/latest/
- **ACCESS Cryosphere Data Pool**: https://access-nri.github.io/ccdtools/